# Visualization Layer Preparation - Gold Layer

Prepares visualization-ready tables for the Streamlit app.

**Inputs:**
- `{catalog}.{gold_schema}.expansion_candidates_h3_enhanced`
- `{catalog}.{silver_schema}.existing_stores_h3`
- `{catalog}.{silver_schema}.pois_competitors`
- `{catalog}.{silver_schema}.isochrones_convenience`
- `{catalog}.{bronze_schema}.census_states`

**Outputs:**
- `{catalog}.{gold_schema}.viz_expansion_candidates` - Normalized scores (0-1)
- `{catalog}.{gold_schema}.viz_existing_stores`
- `{catalog}.{gold_schema}.viz_competitors`
- `{catalog}.{gold_schema}.viz_convenience`
- `{catalog}.{gold_schema}.viz_h3_grid` - H3-8 covering MA boundary

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, expr, explode, lit, when
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", "jdub_demo_aws")
dbutils.widgets.text("bronze_schema", "geo_bronze")
dbutils.widgets.text("silver_schema", "geo_silver")
dbutils.widgets.text("gold_schema", "geo_gold")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")

print(f"Catalog: {catalog}")
print(f"Bronze: {bronze_schema}, Silver: {silver_schema}, Gold: {gold_schema}")

In [ ]:
# MAGIC %md
# MAGIC ## 1. Generate H3 Grid Covering Massachusetts

In [ ]:
# Load Massachusetts boundary
ma_boundary = spark.table(f"{catalog}.{bronze_schema}.census_states").filter(
    (col("state_abbr") == "MA") | (col("state_fips") == "25")
)

# Generate H3-8 grid covering Massachusetts
viz_h3_grid = ma_boundary.select(
    explode(expr("h3_coverash3string(ST_AsBinary(geometry), 5)")).alias("coarse_h3")
).select(
    explode(expr("h3_tochildren(coarse_h3, 8)")).alias("h3_cell_id")
).distinct().withColumn(
    "geometry", expr("ST_GeomFromWKT(h3_boundaryaswkt(h3_cell_id), 4326)")
).withColumn(
    "center_lat", expr("ST_Y(ST_GeomFromWKT(h3_centeraswkt(h3_cell_id), 4326))")
).withColumn(
    "center_lon", expr("ST_X(ST_GeomFromWKT(h3_centeraswkt(h3_cell_id), 4326))")
)

print(f"Generated H3-8 grid with {viz_h3_grid.count()} cells")

# Write H3 grid
viz_h3_grid.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.viz_h3_grid")
print(f"Written to {catalog}.{gold_schema}.viz_h3_grid")

In [ ]:
# MAGIC %md
# MAGIC ## 2. Prepare Expansion Candidates (Normalized Scores)

In [ ]:
# Load expansion candidates
candidates = spark.table(f"{catalog}.{gold_schema}.expansion_candidates_h3_enhanced")

# Calculate min/max for normalization
stats = candidates.agg(
    F.min("predicted_annual_sales").alias("min_sales"),
    F.max("predicted_annual_sales").alias("max_sales"),
    F.min("population").alias("min_pop"),
    F.max("population").alias("max_pop")
).collect()[0]

min_sales, max_sales = stats["min_sales"], stats["max_sales"]
min_pop, max_pop = stats["min_pop"], stats["max_pop"]

# Add normalized scores (0-1)
# Handle division by zero case
if max_sales > min_sales:
    viz_candidates = candidates.withColumn(
        "normalized_sales_score",
        (col("predicted_annual_sales") - lit(min_sales)) / lit(max_sales - min_sales)
    )
else:
    viz_candidates = candidates.withColumn("normalized_sales_score", lit(0.5))

if max_pop > min_pop:
    viz_candidates = viz_candidates.withColumn(
        "normalized_pop_score",
        (col("population") - lit(min_pop)) / lit(max_pop - min_pop)
    )
else:
    viz_candidates = viz_candidates.withColumn("normalized_pop_score", lit(0.5))

viz_candidates = viz_candidates.withColumn(
    "percentile_rank",
    F.percent_rank().over(Window.orderBy("predicted_annual_sales"))
).withColumn(
    "geometry", expr("ST_GeomFromWKT(h3_boundaryaswkt(h3_cell_id), 4326)")
)

print(f"Prepared {viz_candidates.count()} visualization candidates")

# Write
viz_candidates.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.viz_expansion_candidates")
print(f"Written to {catalog}.{gold_schema}.viz_expansion_candidates")

In [ ]:
# MAGIC %md
# MAGIC ## 3. Prepare Existing Stores

In [ ]:
try:
    existing_stores = spark.table(f"{catalog}.{silver_schema}.existing_stores_h3")
    
    viz_existing = existing_stores.select(
        "store_number",
        "latitude",
        "longitude",
        "store_type",
        "city",
        "state",
        "population",
        col("total_poi_count").alias("poi_count") if "total_poi_count" in existing_stores.columns else lit(0).alias("poi_count"),
        "geometry"
    ).withColumn(
        "marker_type", lit("existing_lce")
    )
    
    print(f"Prepared {viz_existing.count()} existing stores")
    
    viz_existing.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.viz_existing_stores")
    print(f"Written to {catalog}.{gold_schema}.viz_existing_stores")
    
except Exception as e:
    print(f"Could not prepare existing stores: {e}")

In [ ]:
# MAGIC %md
# MAGIC ## 4. Prepare Competitors

In [ ]:
try:
    competitors = spark.table(f"{catalog}.{silver_schema}.pois_competitors")
    
    viz_competitors = competitors.select(
        col("poi_id").alias("id"),
        "name",
        "latitude",
        "longitude",
        "poi_category",
        "poi_subcategory",
        "address"
    ).withColumn(
        "marker_type", lit("competitor")
    ).withColumn(
        "geometry", expr("ST_Point(longitude, latitude)")
    )
    
    print(f"Prepared {viz_competitors.count()} competitors")
    
    viz_competitors.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.viz_competitors")
    print(f"Written to {catalog}.{gold_schema}.viz_competitors")
    
except Exception as e:
    print(f"Could not prepare competitors: {e}")

In [ ]:
# MAGIC %md
# MAGIC ## 5. Prepare Convenience Store Isochrones

In [ ]:
try:
    convenience = spark.table(f"{catalog}.{silver_schema}.isochrones_convenience")
    
    viz_convenience = convenience.select(
        col("location_id").alias("id"),
        "store_type",
        "latitude",
        "longitude",
        "city",
        "state",
        "drive_time_minutes",
        "area_sqkm",
        "geometry"
    ).withColumn(
        "marker_type", lit("convenience")
    )
    
    print(f"Prepared {viz_convenience.count()} convenience store isochrones")
    
    viz_convenience.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.viz_convenience")
    print(f"Written to {catalog}.{gold_schema}.viz_convenience")
    
except Exception as e:
    print(f"Could not prepare convenience isochrones: {e}")

In [ ]:
# MAGIC %md
# MAGIC ## Summary

In [ ]:
print("=" * 60)
print("VISUALIZATION LAYER SUMMARY")
print("=" * 60)

viz_tables = [
    "viz_h3_grid",
    "viz_expansion_candidates",
    "viz_existing_stores",
    "viz_competitors",
    "viz_convenience"
]

for table_name in viz_tables:
    try:
        count = spark.table(f"{catalog}.{gold_schema}.{table_name}").count()
        print(f"  {table_name}: {count:,} rows")
    except Exception as e:
        print(f"  {table_name}: NOT AVAILABLE")

print("\n" + "=" * 60)
print("VISUALIZATION LAYER COMPLETE")
print("=" * 60)

## Kepler.gl Map Visualization

In [ ]:
%pip install keplergl --quiet

In [ ]:
from keplergl import KeplerGl
import pandas as pd
import json

# Load top expansion candidates with H3 hexagons
top_n = 10  # Top 10 candidates only

candidates_pd = (
    spark.table(f"{catalog}.{gold_schema}.viz_expansion_candidates")
    .orderBy(F.desc("predicted_annual_sales"))
    .limit(top_n)
    .select("h3_cell_id", "latitude", "longitude", "predicted_annual_sales", 
            "population", "normalized_sales_score", "percentile_rank")
    .toPandas()
)

# Load existing LCE stores (points)
try:
    lce_stores_pd = (
        spark.table(f"{catalog}.{gold_schema}.viz_existing_stores")
        .select("store_number", "latitude", "longitude", "city", "state", "population")
        .toPandas()
    )
except:
    lce_stores_pd = pd.DataFrame()

# Load LCE isochrones (polygons) as GeoJSON
try:
    lce_isochrones_df = spark.table(f"{catalog}.{silver_schema}.isochrones_lce")
    lce_isochrones_gdf = (
        lce_isochrones_df
        .selectExpr(
            "location_id as store_number",
            "ST_AsText(geometry) as geometry_wkt",
            "drive_time_minutes",
            "area_sqkm"
        )
        .toPandas()
    )
    
    # Convert WKT to GeoJSON format for Kepler
    from shapely import wkt
    from shapely.geometry import mapping
    
    lce_isochrones_geojson = {
        "type": "FeatureCollection",
        "features": []
    }
    
    for _, row in lce_isochrones_gdf.iterrows():
        geom = wkt.loads(row['geometry_wkt'])
        feature = {
            "type": "Feature",
            "properties": {
                "store_number": row['store_number'],
                "drive_time_minutes": row['drive_time_minutes'],
                "area_sqkm": row['area_sqkm']
            },
            "geometry": mapping(geom)
        }
        lce_isochrones_geojson["features"].append(feature)
except Exception as e:
    print(f"Could not load LCE isochrones: {e}")
    lce_isochrones_geojson = None

# Load convenience store isochrones (polygons) and filter overlapping
try:
    convenience_isochrones_df = spark.table(f"{catalog}.{silver_schema}.isochrones_convenience")
    convenience_isochrones_gdf = (
        convenience_isochrones_df
        .selectExpr(
            "location_id",
            "store_type",
            "ST_AsText(geometry) as geometry_wkt",
            "drive_time_minutes",
            "latitude",
            "longitude",
            "area_sqkm"
        )
        .toPandas()
    )
    
    # Filter out overlapping convenience isochrones
    from shapely import wkt
    from shapely.geometry import mapping
    
    selected_geometries = []
    convenience_isochrones_geojson = {
        "type": "FeatureCollection",
        "features": []
    }
    
    # Sort by area (smaller first, to prioritize compact trade areas)
    convenience_isochrones_gdf = convenience_isochrones_gdf.sort_values('area_sqkm')
    
    for _, row in convenience_isochrones_gdf.iterrows():
        geom = wkt.loads(row['geometry_wkt'])
        
        # Check if this geometry overlaps with any already selected
        overlaps = False
        for existing_geom in selected_geometries:
            if geom.intersects(existing_geom):
                overlaps = True
                break
        
        # Only add if it doesn't overlap
        if not overlaps:
            selected_geometries.append(geom)
            feature = {
                "type": "Feature",
                "properties": {
                    "location_id": row['location_id'],
                    "store_type": row['store_type'],
                    "drive_time_minutes": row['drive_time_minutes'],
                    "latitude": row['latitude'],
                    "longitude": row['longitude'],
                    "area_sqkm": row['area_sqkm']
                },
                "geometry": mapping(geom)
            }
            convenience_isochrones_geojson["features"].append(feature)
    
    print(f"Filtered convenience stores: {len(convenience_isochrones_gdf)} -> {len(convenience_isochrones_geojson['features'])} (removed {len(convenience_isochrones_gdf) - len(convenience_isochrones_geojson['features'])} overlapping)")
    
except Exception as e:
    print(f"Could not load convenience isochrones: {e}")
    convenience_isochrones_geojson = None

print(f"\nLoaded data for Kepler.gl:")
print(f"  Expansion candidates (H3): {len(candidates_pd)}")
print(f"  LCE stores (points): {len(lce_stores_pd)}")
print(f"  LCE isochrones (polygons): {len(lce_isochrones_geojson['features']) if lce_isochrones_geojson else 0}")
print(f"  Convenience isochrones (polygons, non-overlapping): {len(convenience_isochrones_geojson['features']) if convenience_isochrones_geojson else 0}")

In [ ]:
# Create Kepler map with custom styling
map_config = {
    "version": "v1",
    "config": {
        "mapState": {
            "latitude": 42.4072,
            "longitude": -71.3824,
            "zoom": 8
        }
    }
}

# Initialize Kepler map
kepler_map = KeplerGl(height=800, config=map_config)

# Add expansion candidates (H3 hexagons with red gradient)
if not candidates_pd.empty:
    kepler_map.add_data(data=candidates_pd, name="Expansion Candidates")

# Add LCE store points (orange dots)
if not lce_stores_pd.empty:
    kepler_map.add_data(data=lce_stores_pd, name="LCE Stores")

# Add LCE isochrones (light orange polygons)
if lce_isochrones_geojson:
    kepler_map.add_data(data=lce_isochrones_geojson, name="LCE Trade Areas")

# Add convenience isochrones (blue polygons)
if convenience_isochrones_geojson:
    kepler_map.add_data(data=convenience_isochrones_geojson, name="7-Eleven Trade Areas")

# Display map
kepler_map

### Kepler.gl Layer Styling Guide

**Manual styling in the map above:**

1. **Expansion Candidates** (red gradient by predicted_annual_sales):
   - Layer type: H3
   - H3 Column: `h3_cell_id`
   - Fill Color: Red gradient
   - Color based on: `predicted_annual_sales`
   - Opacity: 60-70%

2. **LCE Stores** (orange dots):
   - Layer type: Point
   - Fill Color: Orange (#FF8C00)
   - Radius: 8-10px

3. **LCE Trade Areas** (light orange polygons):
   - Layer type: Polygon
   - Fill Color: Light Orange (#FFB84D)
   - Opacity: 20-30%
   - Stroke: Orange (#FF8C00)

4. **7-Eleven Trade Areas** (blue polygons):
   - Layer type: Polygon
   - Fill Color: Blue (#4A90E2)
   - Opacity: 20-30%
   - Stroke: Dark Blue (#2E5C8A)

Click the layer icons on the left panel to customize colors, opacity, and filtering options.